In [14]:
from sqlalchemy import Table, Column, Integer, Float, String, Boolean, DateTime, Text, JSON, ForeignKey, MetaData, func, insert, text
from datetime import datetime
from database.session import engine, SessionLocal
from loguru import logger


In [ ]:

# Define the dummy_table structure matching the exact database schema
metadata = MetaData()

dummy_table = Table(
    'dummy_table',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True, nullable=False),
    Column('call_start_ts', Float, nullable=True),  # float8 in DB
    Column('call_end_ts', Float, nullable=True),  # float8 in DB
    Column('chat', JSON, nullable=True),  # jsonb in DB
    Column('provider', Text, nullable=True),
    Column('call_id', Text, nullable=True),
    Column('agent_id', Text, nullable=True),
    Column('duration', Float, nullable=True),  # float8 in DB
    Column('meta_data', JSON, nullable=True),  # jsonb in DB
    Column('recording', Text, nullable=True),
    Column('model', Text, nullable=True),
    Column('cost', Float, nullable=True, quote=True),  # "cost" quoted in DB (reserved keyword)
    Column('json_data', JSON, nullable=True),  # jsonb in DB
    Column('disposition', Text, nullable=True),
    Column('created_at', DateTime(timezone=True), server_default=func.now(), nullable=True),  # timestamptz in DB
    Column('is_disposition_available', Boolean, server_default="false", nullable=True),
    Column('transcript', Text, nullable=True),
    Column('is_transcript_available', Boolean, server_default="false", nullable=True),
    Column('is_duplicate', Boolean, server_default="false", nullable=True),
    Column('client_id', Integer, ForeignKey('client.id'), nullable=False),  # int4 NOT NULL in DB
    Column('campaign_id', Integer, ForeignKey('campaign.id'), nullable=False),  # int4 NOT NULL in DB
    Column('contact_to', String, nullable=True),  # varchar in DB
    Column('contact_from', String, nullable=True),  # varchar in DB
    Column('direction', String, nullable=True),  # varchar in DB
    Column('language', String, nullable=True, quote=True),  # "language" quoted in DB (reserved keyword)
    Column('session_id', String, nullable=True),  # varchar in DB
    Column('call_status', String, nullable=True),  # varchar in DB
    Column('error_message', String, nullable=True),  # varchar in DB
)


def query_sql(query: str):
    """Execute a SQL query and return all results."""
    with SessionLocal() as db:
        result = db.execute(text(query))
        return result.fetchall()


In [16]:

import numpy as np
import json

def transform_json_data(json_data_item, language, created_at, transcript, contact_to, contact_from, direction):
    """
    Transform a single json_data record into the format required for dummy_table.
    
    Args:
        json_data_item: The json_data dictionary from capri_data (may be dict or JSON string)
        language: Language from capri_data
        created_at: Created timestamp from capri_data
        transcript: Transcript from capri_data
        contact_to: Contact to from capri_data
        contact_from: Contact from from capri_data
        direction: Direction from capri_data
    
    Returns:
        Dictionary with transformed data ready for insertion
    """
    if not json_data_item:
        return None
    
    # Ensure json_data_item is a dict (should already be parsed in main loop, but double-check)
    if isinstance(json_data_item, str):
        try:
            json_data_item = json.loads(json_data_item)
        except (json.JSONDecodeError, TypeError):
            logger.warning(f"Could not parse json_data_item as JSON in transform function")
            return None
    
    # Calculate cost from cost_breakdown
    cost = 0.0
    if isinstance(json_data_item.get('cost_breakdown'), list):
        cost = sum([j.get('credit', 0) for j in json_data_item.get('cost_breakdown', [])])
    
    # Get call_end_ts - try different possible paths
    call_end_ts = None
    if 'call_metrics' in json_data_item and isinstance(json_data_item['call_metrics'], dict):
        call_end_ts = json_data_item['call_metrics'].get('call_end_ts')
    elif 'call_end_ts' in json_data_item:
        call_end_ts = json_data_item['call_end_ts']
    
    # Determine if transcript is available
    is_transcript_available = bool(transcript)
    
    # Determine if disposition is available
    disposition = json_data_item.get('disposition')
    is_disposition_available = bool(disposition)
    
    # Extract voip information
    voip = json_data_item.get('voip', {})
    if isinstance(voip, dict):
        provider = voip.get('provider')
        # Use contact_to/contact_from from query if available, otherwise from json_data
        final_contact_to = contact_to if contact_to else voip.get('to')
        final_contact_from = contact_from if contact_from else voip.get('from')
        final_direction = direction if direction else voip.get('direction')
    else:
        provider = None
        final_contact_to = contact_to
        final_contact_from = contact_from
        final_direction = direction
    
    # Extract agent_config information
    agent_config = json_data_item.get('agent_config', {})
    if isinstance(agent_config, dict):
        llm = agent_config.get('llm', {})
        model = llm.get('model') if isinstance(llm, dict) else None
        final_language = language if language else agent_config.get('language')
    else:
        model = None
        final_language = language
    
    # Extract recording URL
    recording = None
    recording_data = json_data_item.get('recording')
    if isinstance(recording_data, dict):
        recording = recording_data.get('recording_url')
    elif isinstance(recording_data, str):
        recording = recording_data
    
    # Handle chat field - ensure it's a dict/list or None, not a string
    chat = json_data_item.get('chat')
    if isinstance(chat, str):
        try:
            chat = json.loads(chat) if chat.lower() != 'null' else None
        except (json.JSONDecodeError, AttributeError):
            chat = None
    
    # Handle meta_data field - ensure it's a dict or None, not a string
    meta_data = json_data_item.get('metadata')
    if isinstance(meta_data, str):
        try:
            meta_data = json.loads(meta_data) if meta_data.lower() != 'null' else None
        except (json.JSONDecodeError, AttributeError):
            meta_data = None
    
    return {
        'call_start_ts': json_data_item.get('ts'),
        'call_end_ts': call_end_ts,
        'chat': chat,
        'provider': provider,
        'call_id': json_data_item.get('call_id'),
        'agent_id': json_data_item.get('agent_id'),
        'duration': json_data_item.get('duration'),
        'meta_data': meta_data,
        'recording': recording,
        'model': model,
        'cost': cost,
        'json_data': json_data_item,  # Keep as dict for JSON column
        'disposition': disposition,
        'created_at': created_at if created_at else datetime.now(),
        'is_disposition_available': is_disposition_available,
        'transcript': transcript,
        'is_transcript_available': is_transcript_available,
        'is_duplicate': False,
        'client_id': np.random.randint(1, 10),  # Random client_id between 1-9
        'campaign_id': np.random.randint(1, 20),  # Random campaign_id between 1-19
        'contact_to': final_contact_to,
        'contact_from': final_contact_from,
        'direction': final_direction,
        'language': final_language,
        'session_id': json_data_item.get('session_id'),
        'call_status': json_data_item.get('call_status'),
        'error_message': json_data_item.get('error_message'),
    }


In [6]:
data = query_sql('''
        SELECT json_data, language, created_at, transcript, contact_to, contact_from, direction 
        FROM capri_data;
    ''')

In [18]:
json_data_list = [i[0] for i in data]

In [8]:
transformed_records = []

In [ ]:
for idx, row in enumerate(data):
            json_data_item = row[0]
            language = row[1]
            created_at = row[2]
            transcript = row[3]
            contact_to = row[4]
            contact_from = row[5]
            direction = row[6]
            
            # Handle case where json_data might be a string from PostgreSQL JSONB
            if isinstance(json_data_item, str):
                try:
                    json_data_item = json.loads(json_data_item)
                except (json.JSONDecodeError, TypeError):
                    logger.warning(f"Row {idx + 1}: Could not parse json_data as JSON, skipping...")
                    continue
            
            if json_data_item:
                transformed = transform_json_data(
                    json_data_item,
                    language,
                    created_at,
                    transcript,
                    contact_to,
                    contact_from,
                    direction
                )
                if transformed:
                    # Ensure client_id is always present and valid
                    if 'client_id' not in transformed or transformed['client_id'] is None:
                        transformed['client_id'] = np.random.randint(1, 10)
                    # Ensure campaign_id is always present and valid
                    if 'campaign_id' not in transformed or transformed['campaign_id'] is None:
                        transformed['campaign_id'] = np.random.randint(1, 20)
                    transformed_records.append(transformed)
            else:
                logger.warning(f"Row {idx + 1} has no json_data, skipping...")
        
        logger.info(f"Transformed {len(transformed_records)} records")
        
        # Validate that all records have required fields
        required_fields = ['client_id', 'campaign_id']
        for idx, record in enumerate(transformed_records[:5]):  # Check first 5 records
            for field in required_fields:
                if field not in record or record[field] is None:
                    logger.error(f"Record {idx + 1} is missing required field: {field}")
                    logger.error(f"Record keys: {list(record.keys())}")
                    raise ValueError(f"Record {idx + 1} is missing required field: {field}")
        
        # Check first record
        if transformed_records:
            logger.info(f"First record keys: {list(transformed_records[0].keys())}")
            logger.info(f"First record client_id: {transformed_records[0].get('client_id')}")
            logger.info(f"First record campaign_id: {transformed_records[0].get('campaign_id')}")
        

In [21]:
transformed_records[0]

{'call_start_ts': 1764592606.957374,
 'call_end_ts': None,
 'chat': None,
 'provider': 'exotel',
 'call_id': '+918047095601:+919351491144',
 'agent_id': '-OfD3XOolKTVGIlmQfcD',
 'duration': None,
 'meta_data': {'Name': 'छोटा राम',
  'DueDateInWords': 'पांच दिसंबर 2025',
  'DaysLeftInWords': '5',
  'EmiAmountInWords': 'नौ हजार चार सौ बयालीस'},
 'recording': None,
 'model': 'gpt-4o-mini',
 'cost': 0.0,
 'json_data': {'ts': 1764592606.957374,
  'voip': {'to': '+919351491144',
   'from': '+918047095601',
   'provider': 'exotel'},
  'call_id': '+918047095601:+919351491144',
  'agent_id': '-OfD3XOolKTVGIlmQfcD',
  'metadata': {'Name': 'छोटा राम',
   'DueDateInWords': 'पांच दिसंबर 2025',
   'DaysLeftInWords': '5',
   'EmiAmountInWords': 'नौ हजार चार सौ बयालीस'},
  'session_id': '-OfOu_nypWyTDXzosbpR',
  'call_status': 'failed',
  'campaign_id': '-OfO_egav5PoJwH7v5_Q',
  'agent_config': {'llm': {'model': 'gpt-4o-mini', 'temperature': 0},
   'flow': {'user_start_first': False,
    'inactivity_h

In [22]:
# Insert records in batches
batch_size = 1
total_inserted = 0


In [ ]:
with engine.connect() as conn:
    for i in range(0, len(transformed_records), batch_size):
        batch = transformed_records[i:i + batch_size]
        
        # Double-check first record in batch has client_id
        if batch and ('client_id' not in batch[0] or batch[0]['client_id'] is None):
            logger.error(f"First record in batch {i // batch_size + 1} is missing client_id")
            logger.error(f"Keys in first record: {list(batch[0].keys())}")
            raise ValueError(f"Batch {i // batch_size + 1} has records without client_id")
        
        # Ensure all records in batch have client_id
        for j, record in enumerate(batch):
            if 'client_id' not in record or record['client_id'] is None:
                record['client_id'] = np.random.randint(1, 10)
            if 'campaign_id' not in record or record['campaign_id'] is None:
                record['campaign_id'] = np.random.randint(1, 20)
        
        stmt = insert(dummy_table)
        result = conn.execute(stmt, batch)
        conn.commit()
        total_inserted += result.rowcount
        logger.info(f"Inserted batch {i // batch_size + 1}: {result.rowcount} rows (Total: {total_inserted}/{len(transformed_records)})")


IntegrityError: (psycopg2.errors.NotNullViolation) null value in column "client_id" of relation "dummy_table" violates not-null constraint
DETAIL:  Failing row contains (1033, 1764592606.957374, null, null, exotel, +918047095601:+919351491144, -OfD3XOolKTVGIlmQfcD, null, {"Name": "छोटा राम", "DueDateInWords": "पां..., null, gpt-4o-mini, 0, {"ts": 1764592606.957374, "voip": {"to": "+919351491144", "from"..., null, 2025-12-02 06:58:47.956408+00, f, null, f, f, null, 9, +919351491144, +918047095601, null, hi, -OfOu_nypWyTDXzosbpR, failed, No response from telephony provider).

[SQL: INSERT INTO dummy_table (call_start_ts, call_end_ts, chat, provider, call_id, agent_id, duration, meta_data, recording, model, cost, json_data, disposition, created_at, is_disposition_available, transcript, is_transcript_available, is_duplicate, campaign_id, contact_to, contact_from, direction, language, session_id, call_status, error_message) VALUES (%(call_start_ts)s, %(call_end_ts)s, %(chat)s::JSON, %(provi